# Notebook 08: ONNX Export

**Module:** 11 Production ML  
**Duration:** ~2 hrs

---

## Learning Objectives

1. Export PyTorch model to ONNX
2. Validate ONNX vs PyTorch outputs
3. Deploy ONNX with ONNX Runtime
4. Plan water-bodies ONNX export

## ONNX (Open Neural Network Exchange)

**Why:** Deploy same model in PyTorch, C++, TensorRT, mobile, browser.

**Export:**
```python
torch.onnx.export(model, dummy_input, 'model.onnx',
    input_names=['image'], output_names=['logits'],
    dynamic_axes={'image': {0: 'batch'}, 'logits': {0: 'batch'}})
```

In [ ]:
import os
import json
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

plt.rcParams['figure.figsize'] = (8, 5)
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
class TinySeg(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(6, 16, 3, padding=1), nn.ReLU(), nn.Conv2d(16, 2, 1))
    def forward(self, x):
        return self.net(x)

model = TinySeg().eval()
dummy = torch.randn(1, 6, 128, 128)
out_pt = model(dummy)

try:
    torch.onnx.export(model, dummy, '/tmp/tiny_seg.onnx', input_names=['image'], output_names=['logits'])
    import onnxruntime as ort
    sess = ort.InferenceSession('/tmp/tiny_seg.onnx')
    out_onnx = sess.run(None, {'image': dummy.numpy()})[0]
    print(f'Max diff PT vs ONNX: {np.abs(out_pt.detach().numpy() - out_onnx).max():.6f}')
except ImportError:
    print('Optional: pip install onnx onnxruntime')

---

## Summary

ONNX enables cross-framework deployment and TensorRT conversion.

**Next:** [09_TensorRT.ipynb](09_TensorRT.ipynb)